# XGBoost Grade/Subgrade and Interest Rate Ablation

This notebook documents the four requested feature-set combinations using the already-preprocessed `missingness_challenger` datasets.

The goal is to make the workflow readable:

1. Load preprocessing outputs.
2. Identify `grade`, `sub_grade`, and `int_rate_clean` features.
3. Build four feature views in memory.
4. Review the neutral XGBoost candidate grid.
5. Display the saved training, validation, test, and economic-policy results.

The full training can be rerun from this notebook, but it is disabled by default because it trains 48 XGBoost candidates.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

## 1. Paths

The ablation starts from preprocessing artifacts. No new preprocessing datasets are required because the feature combinations are created by column filtering from the same preprocessed base tables.

In [2]:
current = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [current, *current.parents]:
    if candidate.name == "CreditRiskRAG":
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    candidate = current / "CreditRiskRAG"
    if candidate.exists():
        PROJECT_ROOT = candidate
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the CreditRiskRAG project root from " + str(current))

DATASET_DIR = PROJECT_ROOT / "Modeling" / "Preprocessing" / "preprocessing_outputs" / "datasets"
FINAL_TABLE_DIR = PROJECT_ROOT / "Modeling" / "modeling_outputs" / "final_comparison" / "tables"
ABLATION_TABLE_DIR = PROJECT_ROOT / "Modeling" / "modeling_outputs" / "xgboost_grade_int_rate_ablation" / "tables"
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "test_xgboost_grade_int_rate_combinations.py"

for path in [DATASET_DIR, FINAL_TABLE_DIR, SCRIPT_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Dataset directory:", DATASET_DIR)
print("Final table directory:", FINAL_TABLE_DIR)
print("Ablation script:", SCRIPT_PATH)

Dataset directory: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/Preprocessing/preprocessing_outputs/datasets
Final table directory: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/Modeling/modeling_outputs/final_comparison/tables
Ablation script: /Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/Final_Project/Final/CreditRiskRAG/scripts/test_xgboost_grade_int_rate_combinations.py


## 2. Load Base Preprocessing Outputs

The base feature matrix is `missingness_challenger`. It includes the missingness indicators created in preprocessing plus encoded categorical features.

In [3]:
def load_parquet(name: str) -> pd.DataFrame:
    path = DATASET_DIR / f"{name}.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path)


X_train_base = load_parquet("missingness_challenger_train_X")
X_validation_base = load_parquet("missingness_challenger_validation_X")
X_test_base = load_parquet("missingness_challenger_test_X")
y_train = load_parquet("train_y")["target_bad"].astype(int)
y_validation = load_parquet("validation_y")["target_bad"].astype(int)
y_test = load_parquet("test_y")["target_bad"].astype(int)

dataset_summary = pd.DataFrame([
    {"split": "train", "rows": len(X_train_base), "features": X_train_base.shape[1], "bad_rate": y_train.mean()},
    {"split": "validation", "rows": len(X_validation_base), "features": X_validation_base.shape[1], "bad_rate": y_validation.mean()},
    {"split": "test", "rows": len(X_test_base), "features": X_test_base.shape[1], "bad_rate": y_test.mean()},
])
dataset_summary["bad_rate"] = dataset_summary["bad_rate"].round(6)
display(dataset_summary)

,split,rows,features,bad_rate
0,train,962641,102,0.188300
1,validation,186920,102,0.246763
2,test,195749,102,0.210315


## 3. Identify Grade/Subgrade and Interest Rate Columns

The model uses one-hot encoded `grade_*` and `sub_grade_*` columns. Interest rate is represented as `int_rate_clean`.

In [4]:
base_columns = list(X_train_base.columns)
grade_subgrade_cols = [c for c in base_columns if c.startswith("grade_") or c.startswith("sub_grade_")]
int_rate_cols = [c for c in base_columns if c == "int_rate_clean"]

feature_group_summary = pd.DataFrame([
    {"feature_group": "grade/sub_grade encoded columns", "count": len(grade_subgrade_cols), "columns": grade_subgrade_cols},
    {"feature_group": "interest rate", "count": len(int_rate_cols), "columns": int_rate_cols},
])
display(feature_group_summary)

,feature_group,count,columns
0,grade/sub_grade encoded columns,42,"[grade_A, grade_B, grade_C, grade_D, grade_E, ..."
1,interest rate,1,[int_rate_clean]


## 4. Define the Four Requested Feature Sets

Each feature set is created by dropping columns from the same preprocessed base matrix. This keeps preprocessing constant and isolates the feature inclusion question.

In [5]:
FEATURE_COMBINATIONS = [
    {
        "feature_set": "with_grade_subgrade_with_int_rate",
        "feature_set_label": "with grade/sub_grade + int_rate",
        "include_grade_subgrade": True,
        "include_int_rate": True,
    },
    {
        "feature_set": "without_grade_subgrade_with_int_rate",
        "feature_set_label": "without grade/sub_grade + int_rate",
        "include_grade_subgrade": False,
        "include_int_rate": True,
    },
    {
        "feature_set": "with_grade_subgrade_without_int_rate",
        "feature_set_label": "with grade/sub_grade without int_rate",
        "include_grade_subgrade": True,
        "include_int_rate": False,
    },
    {
        "feature_set": "without_grade_subgrade_without_int_rate",
        "feature_set_label": "without grade/sub_grade and without int_rate",
        "include_grade_subgrade": False,
        "include_int_rate": False,
    },
]


def columns_for_combination(combo: dict) -> tuple[list[str], list[str]]:
    dropped = []
    if not combo["include_grade_subgrade"]:
        dropped.extend(grade_subgrade_cols)
    if not combo["include_int_rate"]:
        dropped.extend(int_rate_cols)
    dropped = sorted(set(dropped))
    kept = [c for c in base_columns if c not in set(dropped)]
    return kept, dropped


feature_set_rows = []
feature_sets = {}
for combo in FEATURE_COMBINATIONS:
    kept_columns, dropped_columns = columns_for_combination(combo)
    feature_sets[combo["feature_set"]] = {
        "combo": combo,
        "columns": kept_columns,
        "dropped_columns": dropped_columns,
    }
    feature_set_rows.append({
        **combo,
        "kept_feature_count": len(kept_columns),
        "dropped_feature_count": len(dropped_columns),
        "dropped_feature_examples": dropped_columns[:12],
    })

display(pd.DataFrame(feature_set_rows))

,feature_set,feature_set_label,include_grade_subgrade,include_int_rate,kept_feature_count,dropped_feature_count,dropped_feature_examples
0,with_grade_subgrade_with_int_rate,with grade/sub_grade + int_rate,True,True,102,0,[]
1,without_grade_subgrade_with_int_rate,without grade/sub_grade + int_rate,False,True,60,42,"[grade_A, grade_B, grade_C, grade_D, grade_E, ..."
2,with_grade_subgrade_without_int_rate,with grade/sub_grade without int_rate,True,False,101,1,[int_rate_clean]
3,without_grade_subgrade_without_int_rate,without grade/sub_grade and without int_rate,False,False,59,43,"[grade_A, grade_B, grade_C, grade_D, grade_E, ..."


## 5. Create In-Memory Feature Views

These are not saved as new preprocessing datasets. They are pandas views used for this ablation run.

In [6]:
shape_rows = []
for feature_set, config in feature_sets.items():
    cols = config["columns"]
    shape_rows.extend([
        {"feature_set": feature_set, "split": "train", "rows": X_train_base[cols].shape[0], "features": X_train_base[cols].shape[1]},
        {"feature_set": feature_set, "split": "validation", "rows": X_validation_base[cols].shape[0], "features": X_validation_base[cols].shape[1]},
        {"feature_set": feature_set, "split": "test", "rows": X_test_base[cols].shape[0], "features": X_test_base[cols].shape[1]},
    ])

display(pd.DataFrame(shape_rows))

,feature_set,split,rows,features
0,with_grade_subgrade_with_int_rate,train,962641,102
1,with_grade_subgrade_with_int_rate,validation,186920,102
2,with_grade_subgrade_with_int_rate,test,195749,102
3,without_grade_subgrade_with_int_rate,train,962641,60
4,without_grade_subgrade_with_int_rate,validation,186920,60
5,without_grade_subgrade_with_int_rate,test,195749,60
6,with_grade_subgrade_without_int_rate,train,962641,101
7,with_grade_subgrade_without_int_rate,validation,186920,101
8,with_grade_subgrade_without_int_rate,test,195749,101
9,without_grade_subgrade_without_int_rate,train,962641,59


## 6. Modeling Setup

The ablation uses neutral XGBoost:

- `scale_pos_weight=1`
- validation PR-AUC for model selection
- the same candidate grid across all four feature sets
- Platt calibration and a capped economic policy after model selection

In [7]:
PARAM_GRID = [
    {"candidate": "xgb_neutral_01", "params": {"n_estimators": 350, "learning_rate": 0.025, "max_depth": 6, "min_child_weight": 16, "subsample": 0.80, "colsample_bytree": 0.75, "reg_lambda": 5.0, "reg_alpha": 0.2}},
    {"candidate": "xgb_neutral_02", "params": {"n_estimators": 500, "learning_rate": 0.020, "max_depth": 5, "min_child_weight": 16, "subsample": 0.85, "colsample_bytree": 0.80, "reg_lambda": 6.0, "reg_alpha": 0.2}},
    {"candidate": "xgb_neutral_03", "params": {"n_estimators": 550, "learning_rate": 0.018, "max_depth": 4, "min_child_weight": 24, "subsample": 0.90, "colsample_bytree": 0.85, "reg_lambda": 8.0, "reg_alpha": 0.1}},
    {"candidate": "xgb_neutral_04", "params": {"n_estimators": 450, "learning_rate": 0.025, "max_depth": 4, "min_child_weight": 16, "subsample": 0.85, "colsample_bytree": 0.90, "reg_lambda": 5.0, "reg_alpha": 0.0}},
    {"candidate": "xgb_neutral_05", "params": {"n_estimators": 700, "learning_rate": 0.015, "max_depth": 3, "min_child_weight": 24, "subsample": 0.90, "colsample_bytree": 0.90, "reg_lambda": 8.0, "reg_alpha": 0.0}},
    {"candidate": "xgb_neutral_06", "params": {"n_estimators": 400, "learning_rate": 0.030, "max_depth": 5, "min_child_weight": 24, "subsample": 0.80, "colsample_bytree": 0.80, "reg_lambda": 8.0, "reg_alpha": 0.3}},
    {"candidate": "xgb_neutral_07", "params": {"n_estimators": 300, "learning_rate": 0.035, "max_depth": 4, "min_child_weight": 32, "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 10.0, "reg_alpha": 0.3}},
    {"candidate": "xgb_neutral_08", "params": {"n_estimators": 600, "learning_rate": 0.018, "max_depth": 5, "min_child_weight": 32, "subsample": 0.75, "colsample_bytree": 0.75, "reg_lambda": 12.0, "reg_alpha": 0.5}},
    {"candidate": "xgb_neutral_09", "params": {"n_estimators": 450, "learning_rate": 0.020, "max_depth": 6, "min_child_weight": 24, "subsample": 0.75, "colsample_bytree": 0.70, "reg_lambda": 10.0, "reg_alpha": 0.5}},
    {"candidate": "xgb_neutral_10", "params": {"n_estimators": 250, "learning_rate": 0.050, "max_depth": 3, "min_child_weight": 16, "subsample": 0.90, "colsample_bytree": 0.90, "reg_lambda": 4.0, "reg_alpha": 0.0}},
    {"candidate": "xgb_neutral_11", "params": {"n_estimators": 500, "learning_rate": 0.022, "max_depth": 4, "min_child_weight": 8, "subsample": 0.80, "colsample_bytree": 0.85, "reg_lambda": 6.0, "reg_alpha": 0.1}},
    {"candidate": "xgb_neutral_12", "params": {"n_estimators": 650, "learning_rate": 0.016, "max_depth": 4, "min_child_weight": 32, "subsample": 1.00, "colsample_bytree": 0.80, "reg_lambda": 12.0, "reg_alpha": 0.2}},
]

display(pd.DataFrame([
    {"candidate": item["candidate"], **item["params"], "scale_pos_weight": 1.0}
    for item in PARAM_GRID
]))

,candidate,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,scale_pos_weight
0,xgb_neutral_01,350,0.025,6,16,0.80,0.75,5.0,0.2,1.0
1,xgb_neutral_02,500,0.020,5,16,0.85,0.80,6.0,0.2,1.0
2,xgb_neutral_03,550,0.018,4,24,0.90,0.85,8.0,0.1,1.0
3,xgb_neutral_04,450,0.025,4,16,0.85,0.90,5.0,0.0,1.0
4,xgb_neutral_05,700,0.015,3,24,0.90,0.90,8.0,0.0,1.0
5,xgb_neutral_06,400,0.030,5,24,0.80,0.80,8.0,0.3,1.0
6,xgb_neutral_07,300,0.035,4,32,0.85,0.85,10.0,0.3,1.0
7,xgb_neutral_08,600,0.018,5,32,0.75,0.75,12.0,0.5,1.0
8,xgb_neutral_09,450,0.020,6,24,0.75,0.70,10.0,0.5,1.0
9,xgb_neutral_10,250,0.050,3,16,0.90,0.90,4.0,0.0,1.0


## 7. Optional Full Rerun

The output tables already exist from the full run. Set `RUN_FULL_TRAINING = True` only when you want to retrain all 48 candidates.

In [8]:
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    completed = subprocess.run(
        [sys.executable, str(SCRIPT_PATH)],
        cwd=str(PROJECT_ROOT),
        check=True,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
else:
    print("Skipped full training rerun. Loading saved ablation outputs.")

Skipped full training rerun. Loading saved ablation outputs.


## 8. Load Saved Ablation Outputs

In [9]:
OUTPUT_FILES = {
    "feature_summary": FINAL_TABLE_DIR / "xgboost_grade_int_rate_feature_summary.csv",
    "candidate_results": FINAL_TABLE_DIR / "xgboost_grade_int_rate_candidate_results.csv",
    "selected_candidates": FINAL_TABLE_DIR / "xgboost_grade_int_rate_selected_candidates.csv",
    "selected_metrics": FINAL_TABLE_DIR / "xgboost_grade_int_rate_selected_model_metrics.csv",
    "economic_policy": FINAL_TABLE_DIR / "xgboost_grade_int_rate_economic_policy.csv",
}

missing_outputs = [path for path in OUTPUT_FILES.values() if not path.exists()]
if missing_outputs:
    raise FileNotFoundError(
        "Missing ablation outputs. Set RUN_FULL_TRAINING=True above, or run the script first: "
        + ", ".join(str(path) for path in missing_outputs)
    )

feature_summary = pd.read_csv(OUTPUT_FILES["feature_summary"])
candidate_results = pd.read_csv(OUTPUT_FILES["candidate_results"])
selected_candidates = pd.read_csv(OUTPUT_FILES["selected_candidates"])
selected_metrics = pd.read_csv(OUTPUT_FILES["selected_metrics"])
economic_policy = pd.read_csv(OUTPUT_FILES["economic_policy"])

display(feature_summary[[
    "feature_set_label", "include_grade_subgrade", "include_int_rate",
    "kept_feature_count", "dropped_feature_count"
]])

,feature_set_label,include_grade_subgrade,include_int_rate,kept_feature_count,dropped_feature_count
0,with grade/sub_grade + int_rate,True,True,102,0
1,without grade/sub_grade + int_rate,False,True,60,42
2,with grade/sub_grade without int_rate,True,False,101,1
3,without grade/sub_grade and without int_rate,False,False,59,43


## 9. Validation Model Selection

Each feature set is tuned independently, and the best candidate is selected by validation PR-AUC.

In [10]:
display(selected_candidates[[
    "selection_rank", "feature_set_label", "candidate", "feature_count",
    "pr_auc", "roc_auc", "mean_predicted_probability_raw",
    "best_f1_precision", "best_f1_recall"
]])

,selection_rank,feature_set_label,candidate,feature_count,pr_auc,roc_auc,mean_predicted_probability_raw,best_f1_precision,best_f1_recall
0,1,with grade/sub_grade without int_rate,xgb_neutral_01_with_grade_subgrade_without_int...,101,0.426470,0.702002,0.193563,0.351969,0.728997
1,2,without grade/sub_grade + int_rate,xgb_neutral_09_without_grade_subgrade_with_int...,60,0.426423,0.702038,0.199975,0.360500,0.695935
2,3,with grade/sub_grade + int_rate,xgb_neutral_09_with_grade_subgrade_with_int_rate,102,0.426323,0.702142,0.197618,0.364687,0.679935
3,4,without grade/sub_grade and without int_rate,xgb_neutral_06_without_grade_subgrade_without_...,59,0.422215,0.696424,0.189554,0.351418,0.708878


## 10. Validation and Test Metrics

The best-F1 columns remain diagnostics only. The final decision policy is not based on the F1 threshold.

In [11]:
display(selected_metrics[[
    "feature_set_label", "split", "pr_auc", "roc_auc",
    "mean_predicted_probability_raw", "best_f1_predicted_reject_share",
    "review_pct", "review_precision", "review_recall"
]])

,feature_set_label,split,pr_auc,roc_auc,mean_predicted_probability_raw,best_f1_predicted_reject_share,review_pct,review_precision,review_recall
0,with grade/sub_grade + int_rate,validation,0.426323,0.702142,0.197618,0.460074,20.0,0.449658,0.364444
1,with grade/sub_grade + int_rate,test,0.380704,0.710495,0.193062,0.423175,20.0,0.399796,0.380189
2,without grade/sub_grade + int_rate,validation,0.426423,0.702038,0.199975,0.476370,20.0,0.449898,0.364640
3,without grade/sub_grade + int_rate,test,0.380806,0.710201,0.197260,0.420666,20.0,0.400945,0.381282
4,with grade/sub_grade without int_rate,validation,0.426470,0.702002,0.193563,0.511096,20.0,0.449497,0.364314
5,with grade/sub_grade without int_rate,test,0.381304,0.710762,0.188409,0.410536,20.0,0.399770,0.380165
6,without grade/sub_grade and without int_rate,validation,0.422215,0.696424,0.189554,0.497769,20.0,0.447544,0.362732
7,without grade/sub_grade and without int_rate,test,0.375358,0.703070,0.187451,0.433468,20.0,0.396398,0.376958


## 11. Platt-Calibrated Economic Policy

After model selection, the policy uses Platt-calibrated scores and a 20% maximum reject/review cap selected on validation.

In [12]:
display(economic_policy[[
    "feature_set_label", "split", "threshold", "predicted_reject_share",
    "approved_share", "precision_bad_rate_among_rejected",
    "recall_default_capture", "approved_bad_rate",
    "total_portfolio_value", "value_per_applicant"
]])

,feature_set_label,split,threshold,predicted_reject_share,approved_share,precision_bad_rate_among_rejected,recall_default_capture,approved_bad_rate,total_portfolio_value,value_per_applicant
0,with grade/sub_grade + int_rate,validation,0.325248,0.200000,0.800000,0.449658,0.364444,0.196040,137239000.0,734.21
1,with grade/sub_grade + int_rate,test,0.325248,0.201288,0.798712,0.399497,0.382351,0.162638,121918500.0,622.83
2,without grade/sub_grade + int_rate,validation,0.327936,0.200000,0.800000,0.449898,0.364640,0.195980,137342500.0,734.77
3,without grade/sub_grade + int_rate,test,0.327936,0.204854,0.795146,0.399127,0.388763,0.161671,123907500.0,632.99
4,with grade/sub_grade without int_rate,validation,0.328369,0.200000,0.800000,0.449497,0.364314,0.196080,137170000.0,733.84
5,with grade/sub_grade without int_rate,test,0.328369,0.199225,0.800775,0.399867,0.378780,0.163157,120834000.0,617.29
6,without grade/sub_grade and without int_rate,validation,0.321111,0.200000,0.800000,0.447544,0.362732,0.196568,136330500.0,729.35
7,without grade/sub_grade and without int_rate,test,0.321111,0.204859,0.795141,0.394130,0.383905,0.162957,121606000.0,621.23


## 12. Takeaway

The ablation study shows that `grade/sub_grade` and `int_rate_clean` provide highly overlapping information. Models using either signal achieved nearly identical validation PR-AUC, test PR-AUC, reject share, and rejected bad rate. Including both did not improve performance.

Removing both produced the only noticeable decline, indicating that Lending Club's pricing/risk signal is useful, but only one of these variables is needed. Therefore, the preferred production-style model excludes `grade/sub_grade` and keeps `int_rate_clean` for parsimony and equivalent predictive performance.